# Notebook 6: Batch Train All Continual SD-LoRA Adapters

Bu notebook, Notebook 2 akisini baz alir ve sabit immutable GitHub Release'den materyalize edilen adapter datasetlerini sirayla egitir.

Akis:
1. Repo bootstrap ve Notebook 2 helper'larini yukle.
2. Adapter matrisi ve sirayi tanimla.
3. Her adapter icin Notebook 2 parametre, dataset validation, training, OOD calibration, export ve final evaluation hucrelerini calistir.
4. Her run icin ozet yazdir; runtime auto-disconnect batch bitene kadar kapali kalir.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

def _configure_colab_git_read_access():
    token = str(os.environ.get('AADS_GITHUB_RELEASE_READ_TOKEN', '')).strip()
    if not token:
        try:
            from google.colab import userdata
            token = str(userdata.get('AADS_GITHUB_RELEASE_READ_TOKEN') or '').strip()
        except Exception:  # Colab secret access raises provider-specific exceptions.
            token = ''
    os.environ['GIT_TERMINAL_PROMPT'] = '0'
    if not token:
        return
    askpass = Path('/tmp/aads_git_read_askpass.sh')
    askpass.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        "*Username*) printf '%s\\n' 'x-access-token' ;;\n"
        "*) printf '%s\\n' \"$AADS_GIT_READ_TOKEN\" ;;\n"
        'esac\n',
        encoding='utf-8',
    )
    askpass.chmod(0o700)
    os.environ['GIT_ASKPASS'] = str(askpass)
    os.environ['GIT_ASKPASS_REQUIRE'] = 'force'
    os.environ['AADS_GIT_READ_TOKEN'] = token
    os.environ['AADS_GITHUB_RELEASE_READ_TOKEN'] = token


_configure_colab_git_read_access()

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')
NOTEBOOK6_SPARSE_PATHS = (
    'README.md',
    'docs',
    'src',
    'scripts',
    'config',
    'colab_notebooks',
    'requirements.txt',
    'requirements_colab.txt',
    'pyproject.toml',
)



def _ensure_aads_repo_on_path():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, CLONE_TARGET, Path('/content/bitirmeprojesi'), Path('/content/bitirme projesi')]
    for candidate in candidates:
        marker = candidate / 'scripts' / 'notebook_helpers' / 'cell_script_runner.py'
        if marker.is_file():
            repo_root = candidate.resolve()
            if str(repo_root) not in sys.path:
                sys.path.insert(0, str(repo_root))
            return repo_root
    if not CLONE_TARGET.exists():
        subprocess.run(['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse', REPO_URL, str(CLONE_TARGET)], check=True)
        subprocess.run(['git', 'sparse-checkout', 'set', *NOTEBOOK6_SPARSE_PATHS], cwd=str(CLONE_TARGET), check=True)
    if str(CLONE_TARGET) not in sys.path:
        sys.path.insert(0, str(CLONE_TARGET))
    return CLONE_TARGET


_ensure_aads_repo_on_path()

from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell01_bootstrap_access.py', globals())


In [ ]:
NOTEBOOK_NAME = "6_train_all_continual_sd_lora_adapters.ipynb"
NOTEBOOK_FILENAME = "6_train_all_continual_sd_lora_adapters.executed.ipynb"
AUTO_DISCONNECT_RUNTIME = False
AUTO_PUSH_TO_GITHUB = True
NB6_AUTO_DISCONNECT_RUNTIME = True
NB6_AUTO_DISCONNECT_GRACE_SECONDS = 20
DATASET_RELEASE_REPOSITORY = 'EfeErim/bitirmeprojesi'
DATASET_RELEASE_TAG = 'aads-dataset-v1.0.0'
DATASET_RELEASE_CACHE_ROOT = '.runtime_tmp/dataset_release_cache'
ENABLE_BAYESIAN_OPTIMIZATION = True
MANUAL_PARAM_OVERRIDES = {}
DEFAULT_RUNTIME_PARAMS = {
    "AUTO_DISCONNECT_RUNTIME": False,
    "AUTO_PUSH_TO_GITHUB": True,
}
NB6_MANUAL_PARAM_OVERRIDES = {}
NB6_LOW_RESOURCE_MODE = True
NB6_LOW_RESOURCE_OVERRIDES = {
    "ENABLE_BAYESIAN_OPTIMIZATION": False,
    "BATCH_SIZE": 16,
    "GRAD_ACCUM_STEPS": 2,
    "NUM_WORKERS": 2,
    "PREFETCH": 2,
    "PIN_MEMORY": False,
    "USE_CACHE": False,
    "CACHE_TRAIN_SPLIT": False,
    "CHECKPOINT_EVERY_N_STEPS": 0,
}
# Keep empty for the full immutable dataset release; add targets only for explicit research-only runs.
NB6_RESEARCH_ALLOW_UNDER_MIN_ADAPTERS = []

from scripts.notebook_helpers.adapter_recommendations import get_adapter_recs
ADAPTER_RECS = get_adapter_recs()

NB6_ADAPTER_SEQUENCE = [
    "apricot__leaf",
    "apricot__fruit",
    "strawberry__leaf",
    "strawberry__fruit",
    "grape__leaf",
    "grape__fruit",
    "tomato__leaf",
    "tomato__fruit",
]


In [ ]:
import json

from scripts.colab_notebook_helpers import cleanup_notebook_training_state
from src.data.dataset_release_runtime import DatasetReleaseAccessBlocker
from src.pipeline.adapter_release import resolve_token

if not str(resolve_token(write=False) or "").strip():
    raise DatasetReleaseAccessBlocker(
        "DATASET_RELEASE_ACCESS_BLOCKER: Add AADS_GITHUB_RELEASE_READ_TOKEN under Colab > Secrets, "
        "enable notebook access for that secret, then rerun Notebook 6 from the first cell."
    )
print("[NB6][PREFLIGHT] Private dataset Release read token is ready.")

NB6_RESULTS = {}

for index, adapter_key in enumerate(NB6_ADAPTER_SEQUENCE, start=1):
    print(f"\n[NB6] Starting {index}/{len(NB6_ADAPTER_SEQUENCE)}: {adapter_key}")
    adapter_rec = ADAPTER_RECS[adapter_key]
    research_under_min_bypass = adapter_key in NB6_RESEARCH_ALLOW_UNDER_MIN_ADAPTERS
    if research_under_min_bypass:
        adapter_rec["allow_under_min"] = True
        print("[NB6][RESEARCH-ONLY] 100 image/class production floor bypass enabled.")
    CROP_NAME = adapter_rec["crop"]
    PART_NAME = adapter_rec["part"]
    DATASET_NAME = adapter_key
    ADAPTER_KEY = adapter_key
    MANUAL_PARAM_OVERRIDES = dict(NB6_LOW_RESOURCE_OVERRIDES if NB6_LOW_RESOURCE_MODE else {})
    MANUAL_PARAM_OVERRIDES.update(NB6_MANUAL_PARAM_OVERRIDES.get(adapter_key, {}))
    try:
        run_cell_script('nb2_cell03_runtime_setup.py', globals())
        run_cell_script('nb2_cell04_parameter_resolution.py', globals())
        run_cell_script('nb2_cell05_access_check.py', globals())
        run_cell_script('nb2_cell06_dataset_validation.py', globals())
        run_cell_script('nb2_cell07_engine_init.py', globals())
        run_cell_script('nb2_cell08_ood_config_verify.py', globals())
        run_cell_script('nb2_cell09_training.py', globals())
        run_cell_script('nb2_cell10_ood_calibration.py', globals())
        run_cell_script('nb2_cell11_adapter_save.py', globals())
        run_cell_script('nb2_cell12_final_evaluation.py', globals())
        NB6_RESULTS[adapter_key] = {
            "status": "ok",
            "run_id": str(globals().get("RUN_ID", "")),
            "crop": str(globals().get("CROP_NAME", "")),
            "part": str(globals().get("PART_NAME", "")),
            "research_under_min_bypass": bool(research_under_min_bypass),
        }
    except Exception as exc:
        NB6_RESULTS[adapter_key] = {
            "status": "failed",
            "error": str(exc),
            "run_id": str(globals().get("RUN_ID", "")),
            "crop": str(globals().get("CROP_NAME", "")),
            "part": str(globals().get("PART_NAME", "")),
            "research_under_min_bypass": bool(research_under_min_bypass),
        }
        print(f"[NB6] FAILED {adapter_key}: {exc}")
    finally:
        cleanup_notebook_training_state(globals(), label=adapter_key)

print('\n[NB6] SUMMARY')
print(json.dumps(NB6_RESULTS, indent=2, ensure_ascii=False))

from scripts.colab_notebook_helpers import maybe_auto_disconnect_colab_runtime

NB6_FAILED_ADAPTERS = [key for key, result in NB6_RESULTS.items() if result.get("status") != "ok"]
NB6_ALL_ADAPTERS_ATTEMPTED = len(NB6_RESULTS) == len(NB6_ADAPTER_SEQUENCE)
NB6_ALL_ADAPTERS_SUCCEEDED = NB6_ALL_ADAPTERS_ATTEMPTED and not NB6_FAILED_ADAPTERS
NB6_COMPLETION_REPORT = {
    "ready": NB6_ALL_ADAPTERS_SUCCEEDED,
    "checks": {
        "batch_loop_completed": True,
        "all_adapters_attempted": NB6_ALL_ADAPTERS_ATTEMPTED,
        "all_adapters_succeeded": NB6_ALL_ADAPTERS_SUCCEEDED,
    },
    "missing": NB6_FAILED_ADAPTERS,
    "soft_missing": [],
    "adapter_results": NB6_RESULTS,
}
maybe_auto_disconnect_colab_runtime(
    enabled=bool(NB6_AUTO_DISCONNECT_RUNTIME),
    grace_period_sec=float(NB6_AUTO_DISCONNECT_GRACE_SECONDS),
    telemetry=globals().get("TELEMETRY"),
    completion_report=NB6_COMPLETION_REPORT,
)
